### Start with a label and all 8 features. Iterate through 7/8 feature combinations to find the feature with least impact. Drop the feature and iterate to the next 6/7 features... continue until F1 score is not improving anymore

In [1]:
from import_data import data_import
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from classifier import Classifier
import matplotlib.pyplot as plt
import pandas as pd

### Function that takes all features, the label for which we are looking, the features to be dropped and the best F1 from the last rotation. 

1) Create list of active features by dropping the low F1 features from the feature list.
2) Iterate through the active features
2) Import the data and drop the features that are no longer used.
3) Change labels to binary for one vs. all classifier
4) Classify using the classifier function which returns the metrics and conf. matrix.



In [2]:
files = ["SNR_noiseless"]

# Define a function that finds the best F1 score while iterating through n-1/n features and returns the feature to drop
def find_best_feature_to_drop(features: list, target_label: str, feature_dropped: list, best_f1_n0: float):
    """
    Finds the best feature to drop based on the F1 score.

    :param features: list of features to use for classification
    :param target_label: the label to classify against all other labels
    :param feature_dropped: list of features that have already been dropped
    :param best_f1_n0: the best F1 score from the previous iteration (with n features)
    
    :return: a tuple containing:
    1. the feature that, when dropped, gives the best F1 score
    2. the best F1 score achieved by dropping a feature
    """
    active_features = [f for f in features if f not in feature_dropped]
    best_feature = None
    best_f1_local = 0

    for feature in active_features:
        # drop the feature n
        features_to_use = [f for f in active_features if f != feature]

        data, sample_labels = data_import(files, features_to_use)
        full_data = pd.DataFrame(data, columns=features_to_use)

        # Change labels to binary classification: 1 for the target class, 0 for all other events
        binary_labels = [int(sample_label == target_label) for sample_label in sample_labels]

        # add labels to the dataframe
        full_data['Label'] = binary_labels

        X_train, X_test, y_train, y_test = train_test_split(data, binary_labels, test_size=0.2, random_state=42, stratify=binary_labels)

        logistic_regression = LogisticRegression(class_weight='balanced')

        metrics = Classifier(logistic_regression, X_train, y_train, X_test, y_test, title="Logistic Regression - " + target_label + " - Feature not used: " + str(feature), graph=False)
        # find the best F1 score and the corresponding feature to drop
        if metrics["F1 Score"] > best_f1_local:  
            best_f1_local = metrics["F1 Score"]
            best_feature = feature
    # check if the best F1 score is better than the previous best F1 score (best_f1_n0) with a small substraction to account for performance improvements that are not significant
    if best_f1_local > best_f1_n0 - 0.01:
        best_f1 = best_f1_local
    else:
        best_f1 = None  # If no improvement, return None to indicate that dropping any feature does not improve F1 score

    return best_feature, best_f1

### Implementing a Loop that cycles through all labels to determine the ideal features.

In [5]:
# Create a DataFrame to store the ideal features for each target label
ideal_features = pd.DataFrame(columns=["Target Label", "Best Feature to Drop", "F1 Score", "Features to Use"])

# Create a list to store the results for each target label (Append to this list and then convert to DataFrame at the end)
rows = []

target_labels = ["Sag", "Sag_harmonics", "Transient", "Swell_harmonics", "Normal", "Flicker", "Harmonics", "Swell", "Interruption"]
features = ["F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8"]

# Iterate through each target label and find the best features to drop
for target_label in target_labels:
    feature_dropped = []
    best_f1_n0 = 0 # Initial value for the global best F1 score with all features

    while best_f1_n0 is not None and len(feature_dropped) < 7:  # Ensure we don't drop all features
        best_feature_to_drop, best_f1 = find_best_feature_to_drop(features, target_label, feature_dropped, best_f1_n0)

        if best_f1 is not None:
            feature_dropped.append(best_feature_to_drop)
            best_f1_n0 = best_f1
        else:
            print(f"No further improvement in F1 score of {target_label}. Stopping the feature dropping process.")
            print(f"Final features dropped: {feature_dropped}, Final best F1 score: {best_f1_n0}")
            break
    rows.append({"Target Label": target_label, "Best Feature to Drop": feature_dropped, "F1 Score": best_f1_n0, "Features to Use": [f for f in features if f not in feature_dropped]})
ideal_features = pd.DataFrame(rows)
print(ideal_features)

No further improvement in F1 score of Sag. Stopping the feature dropping process.
Final features dropped: ['F4', 'F1', 'F2', 'F6', 'F7'], Final best F1 score: 0.9763559721946252
No further improvement in F1 score of Sag_harmonics. Stopping the feature dropping process.
Final features dropped: ['F1', 'F3', 'F8', 'F4'], Final best F1 score: 0.9171796201144655
No further improvement in F1 score of Transient. Stopping the feature dropping process.
Final features dropped: ['F1', 'F3', 'F5', 'F2', 'F8', 'F4'], Final best F1 score: 0.9981352066390476
No further improvement in F1 score of Swell_harmonics. Stopping the feature dropping process.
Final features dropped: ['F1', 'F8', 'F2', 'F6', 'F5'], Final best F1 score: 0.9657092085065103
No further improvement in F1 score of Normal. Stopping the feature dropping process.
Final features dropped: ['F6', 'F3', 'F5', 'F2', 'F8', 'F1'], Final best F1 score: 0.9199255079099479
No further improvement in F1 score of Flicker. Stopping the feature dropp

In [4]:
# from itertools import combinations
# import pandas as pd
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import cross_val_score

# # 1. Definieren Sie Ihre Feature-Namen
# feature_cols = ["F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8"]

# data, sample_labels = data_import(files, feature_cols)
# full_data = pd.DataFrame(data, columns=feature_cols)

# # Change labels to binary classification: 1 for the target class, 0 for all other events
# binary_labels = [int(sample_label == target_label) for sample_label in sample_labels]

# # add labels to the dataframe
# full_data['Label'] = binary_labels

# X_train, X_test, y_train, y_test = train_test_split(data, binary_labels, test_size=0.2, random_state=42, stratify=binary_labels)


# model = LogisticRegression(class_weight="balanced")
# results = []

# n_features = X_train.shape[1]  # Anzahl der Spalten (z. B. 8)

# for k in range(1, n_features + 1):
#     # Kombinationen aus Integer-Indizes (0, 1, 2, ..., 7) erzeugen
#     for combo in combinations(range(n_features), k):
#         combo_list = list(combo)
        
#         # Datensatz auf die aktuelle Feature-Kombination reduzieren
#         X_subset = X_train[:, combo_list]

#         # Cross-Validation durchführen (z. B. mit F1-Score)
#         scores = cross_val_score(
#             model, X_subset, y_train, cv=3, scoring="f1_weighted"
#         )

#         results.append(
#             {
#                 "num_features": k,
#                 "features": combo_list,
#                 "mean_f1": scores.mean(),
#                 "std_f1": scores.std(),
#             }
#         )

# # 3. Ergebnisse in ein DataFrame umwandeln und nach Performance sortieren
# df_results = pd.DataFrame(results).sort_values(by="mean_f1", ascending=False)

# # Beste Kombination anzeigen
# print("Beste Kombination:")
# print(df_results.iloc[0])